### RapidFire AI RAG/Context Engineering: RapidFire AI Documents Q&A

In [49]:
# import getpass
# TRITON_API_KEY = getpass.getpass("Enter your Triton API key: ")

# The above option is probably safer
from pathlib import Path
TRITON_API_KEY = Path("~/Project1/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

In [50]:
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)

from rapidfireai import Experiment

import re, json, math
from typing import List as listtype, Dict, Any

# project1_eval.py must be in the same directory as this notebook.
# It is shared across evaluate_retrieval.py, run_judge.py, and this notebook;
# do not modify the file itself.
from eval_scripts.project1_eval import (
    to_spans,           # validates + coerces (file, start, end) tuples
    precision_at_k,     # overlap-aware Precision@k
    recall_at_k,        # overlap-aware Recall@k
    f1_at_k,            # harmonic mean of the above
    call_judge,         # LLM-as-judge: correctness / faithfulness / completeness
)


In [51]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"

In [52]:
# Load TensorBoard extension
# %load_ext tensorboard

# TensorBoard log directory will be auto-created in experiment path

##### ⚠️ API Cost Considerations
This notebook runs 4 configurations concurrently on a downsampled dataset of 256 examples.
Estimated Costs:
- Current run (downsampled): \$5 
- Full set: \$45

> 💡 **Tip:** Monitor your API usage to avoid unexpected charges.

### Load Queries to `qrels`

In [53]:
import pandas as pd
from datasets import Dataset
from pathlib import Path

# Load golden QA pairs from validation set
validation_file = Path("~/Project1/validation-set-golden-qa-pairs.json").expanduser()
# test on self generated QA pairs
# validation_file = Path("/home/yup031/Project1/Golden QA Pair Generation/[FINAL] Golden_QA_Pairs.json").expanduser()
with open(validation_file, "r") as f:
    qa_pairs = json.load(f)

# Build queries dataset.
# reference_answer is passed through the batch so postprocess_fn can read it
# without reloading the file on every batch.
queries = {
    "query":            [qa["question"]         for qa in qa_pairs],
    "query_id":         [str(qa["question_id"]) for qa in qa_pairs],
    "reference_answer": [qa["reference_answer"] for qa in qa_pairs],
}
queries = Dataset.from_dict(queries).shuffle(seed=42)

# Build qrels: one row per (query_id, evidence span).
# lines is normalised to a strict 2-element [start_line, end_line] list so it
# is always compatible with project1_eval.to_spans({file, lines: [s, e]}).
qrels_data = []
for qa in qa_pairs:
    query_id = str(qa["question_id"])
    for evidence in qa["source_evidence"]:
        raw_lines = evidence.get("lines", [])
        if len(raw_lines) >= 2:
            start_line, end_line = int(min(raw_lines)), int(max(raw_lines))
        elif len(raw_lines) == 1:
            start_line = end_line = int(raw_lines[0])
        else:
            # No line info — skip; a span with unknown bounds cannot be evaluated
            continue
        qrels_data.append({
            "query_id":   query_id,
            "file":       Path(evidence.get("file", "")).name,
            "start_line": start_line,
            "end_line":   end_line,
            "relevance":  1,
        })

qrels = pd.DataFrame(qrels_data)
print(f"Loaded {len(qa_pairs)} QA pairs")
print(f"Dataset shape: {queries.shape}")
print(f"Qrels shape:   {qrels.shape}")
qrels.head(10)


Loaded 45 QA pairs
Dataset shape: (45, 3)
Qrels shape:   (52, 5)


,query_id,file,start_line,end_line,relevance
0,1,configs.rst,23,48,1
1,2,difference.rst,4,29,1
2,3,experiment.rst,27,39,1
3,4,configs.rst,58,102,1
4,5,walkthroughrag.rst,29,103,1
5,6,icops.rst,75,176,1
6,7,evalsfunctions.rst,8,31,1
7,8,ragspecs.rst,56,98,1
8,8,rag_fiqa.rst,49,59,1
9,9,experiment.rst,215,219,1


### Create Experiment

In [54]:
# from rapidfireai import Experiment
my_experiment = "exp_0508_1"
experiment = Experiment(experiment_name=my_experiment, mode="evals")

The previously running experiment exp_0508_2 was forcibly ended. Created a new experiment 'exp_0508_3' with Experiment ID: 104 at /home/yup031/rapidfireai/rapidfire_experiments/exp_0508_3


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# Chunking utilities
#
# Two splitter wrappers are provided for comparison in the grid search:
#
#   LineNumberSplitter      – wraps RecursiveCharacterTextSplitter.
#                             Passes add_start_index=True to the inner splitter
#                             so char offsets are available, then tags each chunk
#                             with span_coord="char" so compute_metrics_fn knows
#                             to convert char offsets → line numbers at eval time.
#
#   StructuralRSTSplitter   – wraps rst_structural_splitter.
#                             Splits on RST structural boundaries and writes
#                             start_line/end_line natively. Tags each chunk with
#                             span_coord="line" so no conversion is needed.
#
# Metadata written per chunk (both strategies)
# --------------------------------------------
#   source      str  – absolute path (set by TextLoader)
#   span_coord  str  – "char" | "line"  (consumed by compute_metrics_fn)
#
# Additionally per strategy:
#   LineNumberSplitter    → start_index, end_index  (char offsets)
#   StructuralRSTSplitter → start_line, end_line    (1-based line numbers)
# ─────────────────────────────────────────────────────────────────────────────

import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import TextSplitter


# ── rst_structural_splitter ──────────────────────────────────────────────────

_RST_HEADING_RE = re.compile(r"^([=\-~^'`#*+])(\1*)$")

def rst_structural_splitter(
    documents: list[Document],
    min_tokens: int = 64,
    max_tokens: int = 256,
    tokenizer_encoding: str = "gpt2",
) -> list[Document]:
    """Split RST documents on structural boundaries (sections, paragraphs).

    Writes start_line / end_line (1-based, inclusive) directly into metadata.
    Falls back to sentence splitting for paragraphs that exceed max_tokens.
    """
    import tiktoken
    enc = tiktoken.get_encoding(tokenizer_encoding)

    def token_len(text: str) -> int:
        return len(enc.encode(text))

    chunks: list[Document] = []

    for doc in documents:
        src   = doc.metadata.get("source", "")
        lines = doc.page_content.splitlines(keepends=True)

        # pass 1: locate section boundaries
        section_starts = [0]
        for i, line in enumerate(lines):
            stripped = line.rstrip("\n")
            if _RST_HEADING_RE.match(stripped) and len(stripped) >= 2:
                title_idx = max(0, i - 1)
                if title_idx not in section_starts:
                    section_starts.append(title_idx)

        section_starts = sorted(set(section_starts))
        section_ranges = list(zip(section_starts, section_starts[1:] + [len(lines)]))

        for sec_start, sec_end in section_ranges:
            section_lines = lines[sec_start:sec_end]

            # pass 2: split into paragraphs on blank lines
            paragraphs: list[list[tuple[int, str]]] = []
            current:    list[tuple[int, str]]        = []
            for offset, line in enumerate(section_lines):
                lineno = sec_start + offset + 1  # 1-based
                if line == "" and current:
                    paragraphs.append(current)
                    current = []
                else:
                    current.append((lineno, line))
            if current:
                paragraphs.append(current)

            # pass 3: merge tiny paragraphs; split oversized ones
            # flush is defined outside the section loop to avoid closure capture issues
            def _flush(
                buf: list[tuple[int, str]],
                _src: str,
                chunk_type: str = "paragraph",
            ) -> None:
                if not buf:
                    return
                text = "".join(t for _, t in buf).strip()
                if not text:
                    return
                chunks.append(Document(
                    page_content=text,
                    metadata={
                        "source":     _src,
                        "start_line": buf[0][0],
                        "end_line":   buf[-1][0],
                        "chunk_type": chunk_type,
                        "span_coord": "line",
                    },
                ))

            buffer:    list[tuple[int, str]] = []
            prev_para: list[tuple[int, str]] = []   

            for para in paragraphs:
                para_text   = "".join(t for _, t in para).strip()
                para_tokens = token_len(para_text)

                if para_tokens > max_tokens:
                    _flush(buffer, src)
                    buffer    = []
                    prev_para = []                 
                    line_buf:  list[tuple[int, str]] = []
                    line_toks  = 0
                    for lineno, line_text in para:
                        t = token_len(line_text)
                        if line_toks + t > max_tokens and line_buf:
                            _flush(line_buf, src, chunk_type="overflow")
                            line_buf  = []
                            line_toks = 0
                        line_buf.append((lineno, line_text))
                        line_toks += t
                    _flush(line_buf, src, chunk_type="overflow")

                elif token_len("".join(t for _, t in buffer)) + para_tokens < min_tokens:
                    buffer.extend(para)
                    prev_para = para                

                else:
                    _flush(buffer, src)
                    buffer    = prev_para[:] + para[:]   # overlap one paragraph
                    prev_para = para                    

            _flush(buffer, src)

    return chunks


# ── Splitter wrappers ────────────────────────────────────────────────────────

class LineNumberSplitter(TextSplitter):
    """Wraps RecursiveCharacterTextSplitter; tags chunks with span_coord="char".

    Char offsets (start_index / end_index) are left in metadata as-is.
    compute_metrics_fn reads span_coord and converts them to line numbers
    at eval time using the source files.
    """
    def __init__(self, inner_splitter: TextSplitter):
        # chunk_size/overlap required by TextSplitter.__init__; read from inner.
        super().__init__(chunk_size=inner_splitter._chunk_size,
                        chunk_overlap=inner_splitter._chunk_overlap)
        self._inner = inner_splitter

    def split_documents(self, documents: list[Document]) -> list[Document]:
        chunks = self._inner.split_documents(documents)
        for doc in chunks:
            doc.metadata["span_coord"] = "char"
        return chunks

    def split_text(self, text: str) -> list[str]:
        return self._inner.split_text(text)


class StructuralRSTSplitter(TextSplitter):
    """Wraps rst_structural_splitter; chunks already have span_coord="line"."""

    def __init__(self, min_tokens: int = 64, max_tokens: int =256):
        super().__init__(chunk_size=max_tokens, chunk_overlap=0)
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens

    def split_documents(self, documents: list[Document]) -> list[Document]:
        return rst_structural_splitter(
            documents,
            min_tokens=self.min_tokens,
            max_tokens=self.max_tokens,
        )

    def split_text(self, text: str) -> list[str]:
        doc = Document(page_content=text, metadata={"source": ""})
        return [c.page_content for c in rst_structural_splitter(
            [doc], min_tokens=self.min_tokens, max_tokens=self.max_tokens)]


----- CONFIGS ------

In [56]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, SpacyTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from typing import Dict

batch_size = 32

# def custom_template(retrieved_docs: listtype[Document]) -> str:
#     """Serialise retrieved chunks into a single context string.
#     Plain text only; no metadata injected so the prompt stays clean.
#     """
#     return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Shared config reused across all RAG variants
_LOADER = DirectoryLoader(
    path="/home/yup031/Project1/sourcedocs/",
    glob="*.rst",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    sample_seed=1337,
)
_EMBEDDING_CFG = List([{
    "class": OpenAIEmbeddings,
    "model": "api-tgpt-embeddings",
    "api_key": TRITON_API_KEY,
    "base_url": "https://tritonai-api.ucsd.edu",
    "check_embedding_ctx_length": False,
    }
    # {
    # "class": HuggingFaceEmbeddings,
    # "model_name": "BAAI/bge-large-en-v1.5",
    # "model_kwargs": {"device": "cpu"},
    # "encode_kwargs": {"normalize_embeddings": True}
    # }
    ])
_VECTOR_STORE_CFG = {"type": "faiss", "batch_size": batch_size}
_SEARCH_CFG       = List([{"type": "similarity", "k": 20}])
_RERANKER_CFG     = List([
    # {
    # "class": CrossEncoderReranker,
    # "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
    # "model_kwargs": {"device": "cpu"},
    # "top_n": 5,
    # },
    {
    "class": CrossEncoderReranker,
    "model_name": "BAAI/bge-reranker-v2-m3",
    "model_kwargs": {"device": "cpu"},
    "top_n": 3,
    },
    # {
    # "class": CrossEncoderReranker,
    # "model_name": "BAAI/bge-reranker-v2-m3",
    # "model_kwargs": {"device": "cpu"},
    # "top_n": 5,
    # }
    # {
    # "class": CrossEncoderReranker,
    # "model_name": "mixedbread-ai/mxbai-rerank-large-v1",
    # "model_kwargs": {"device": "cpu"},
    # "top_n": 3,
    # }
    ])

# ─────────────────────────────────────────────────────────────────────────────
# Splitter Configuration: (label, splitter_instance) tuples
# Add new strategies here by appending to _SPLITTER_CONFIGS
# ─────────────────────────────────────────────────────────────────────────────

_SPLITTER_CONFIGS = [
    # ── Strategy A: token-based, chunk_size=256 ──────────────────────────────────
    # LineNumberSplitter wraps the inner splitter and tags each chunk with
    # span_coord="char". Char offsets are converted to line numbers in
    # compute_metrics_fn when to_spans is called.
    # (
    #     "token-256",
    #     LineNumberSplitter(
    #         RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    #             encoding_name="gpt2",
    #             chunk_size=256,
    #             chunk_overlap=32,
    #             add_start_index=True,  # populates start_index / end_index
    #         )
    #     ),
    # ),

    # ── Strategy B: token-based, chunk_size=128 ──────────────────────────────────
    (
        "token-128-overlap-32",
        LineNumberSplitter(
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2",
                chunk_size=128,
                chunk_overlap=32,
                add_start_index=True,
            )
        ),
    ),

    (
        "token-256-overlap128",
        LineNumberSplitter(
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2",
                chunk_size=256,
                chunk_overlap=128,
                add_start_index=True,
            )
        ),
    ),

    # (
    #     "token-512-overlap-256",
    #     LineNumberSplitter(
    #         RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    #             encoding_name="gpt2",
    #             chunk_size=512,
    #             chunk_overlap=256,
    #             add_start_index=True,
    #         )
    #     ),
    # ),


    # ── Strategy C: structure-aware RST splitter ─────────────────────────────────
    # StructuralRSTSplitter writes start_line/end_line natively and tags chunks
    # with span_coord="line", so no conversion is needed at eval time.
    # (
    #     "rst-structural_128",
    #     StructuralRSTSplitter(min_tokens=64, max_tokens=128),
    # ),
    (
        "rst-structural_256",
        StructuralRSTSplitter(min_tokens=128, max_tokens=256),
    ),
    # (
    #     "rst-structural_512",
    #     StructuralRSTSplitter(min_tokens=256, max_tokens=512),
    # )
]

# Build RAG specs from splitter configurations
all_splitters = [
    (
        RFLangChainRagSpec(
            document_loader=_LOADER,
            text_splitter=splitter,
            embedding_cfg=_EMBEDDING_CFG,
            vector_store_cfg=_VECTOR_STORE_CFG,
            search_cfg=_SEARCH_CFG,
            reranker_cfg=None,  #_RERANKER_CFG,
            enable_gpu_search=False,
            # document_template=custom_template,
        ),
        label,
    )
    for label, splitter in _SPLITTER_CONFIGS
]

### Define Data Processing and Postprocessing Functions

In [57]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Retrieve context and build generator prompts.

    Adds to batch
    -------------
    retrieved_spans    list[list[tuple[str, int, int, str]]]
        Per query, rank-ordered (file, start, end, span_coord) tuples.
        start/end are char offsets when span_coord="char" (token splitters),
        or 1-based line numbers when span_coord="line" (structural splitter).
        Conversion to line numbers and to_spans validation happen in
        compute_metrics_fn, not here.
    serialized_context list[str]
        The exact text string passed to the generator for each query.
    prompts            list[list[dict]]
        OpenAI-style message list for each query.
    """
    import tiktoken
    from pathlib import Path

    SYSTEM_INSTRUCTIONS = (
        "You are a technical QA assistant for RapidFire AI documentation. "
        "Answer the question using ONLY the provided context. "
        "Your answer will be scored on three criteria: "
        "Correctness (matches the reference answer), "
        "Faithfulness (no information beyond the context), and "
        "Completeness (covers all key points in the context relevant to the question). "
        "You must provide the answer that best meets all three criteria."

        # "You are a technical QA assistant for RapidFire AI documentation. "
        # "Answer the question using ONLY information present in the provided context. "
        # "Structure your answer as follows: "
        # "1. Directly answer the question. "
        # "2. Include all relevant details, parameters, and examples from the context. "
        # "3. If the context contains multiple relevant sections, synthesize them. "
        # "Do not add information not present in the context, "
        # "but use all relevant information that is present."
    )
    MAX_TOKENS = 2000

    # Batched retrieval → list[list[Document]]
    # all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # ── TRY: sort based on weighted scores of documents + in-doc rerank ────────────────────────────────────────
    all_context_raw = rag.get_context(batch_queries=batch["query"], serialize=False)

    from collections import defaultdict
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder

    _reranker = HuggingFaceCrossEncoder(
        model_name="BAAI/bge-reranker-v2-m3",
        model_kwargs={"device": "cpu"},
    )

    def _select_best_file_and_rerank(query: str, docs: list) -> list:
        if not docs:
            return docs

        # Step 1: score all docs with reranker
        pairs  = [(query, doc.page_content) for doc in docs]
        scores = _reranker.score(pairs)
        scored_docs = list(zip(scores, docs))

        # Step 2: aggregate score per file (weighted sum by rank position)
        file_scores: dict[str, float] = defaultdict(float)
        for rank, (score, doc) in enumerate(
            sorted(scored_docs, key=lambda x: x[0], reverse=True)
        ):
            fname = Path(doc.metadata.get("source", "")).name
            # higher rank = higher weight; score itself also contributes
            file_scores[fname] += score * (1.0 / (rank + 1))

        # Step 3: pick best file
        sorted_files = sorted(file_scores.items(), key=lambda x: x[1], reverse=True)
        best_score   = sorted_files[0][1]
        best_file = {
            f for f, s in sorted_files[:2]
            if s >= best_score * 0.8    # --- adjust the threshold here --------------------------------------
        }

        # Step 4: rerank only within best file, return top-3
        best_file_docs = [
            (score, doc) for score, doc in scored_docs
            if Path(doc.metadata.get("source", "")).name in best_file
        ]
        best_file_docs.sort(key=lambda x: x[0], reverse=True)
        return [doc for _, doc in best_file_docs[:3]]   # adjust the real k here based on preference (top-3 or top-5)------------

    all_context = [
        _select_best_file_and_rerank(query, docs)
        for query, docs in zip(batch["query"], all_context_raw)
    ]

    # Store raw (file, start, end, span_coord) — no conversion or validation here.
    # span_coord is written by the splitter wrapper (LineNumberSplitter or
    # StructuralRSTSplitter) and tells compute_metrics_fn which coordinate system
    # the start/end values are in.
    retrieved_spans = [
        [
            (
                Path(doc.metadata.get("source", "")).name,
                doc.metadata.get("start_index", doc.metadata.get("start_line", 0)),
                # modify to fix the issue of end_index sometimes being missing when add_start_index=True
                doc.metadata.get("start_index", 0) + len(doc.page_content)
                    if doc.metadata.get("span_coord", "char") == "char"
                    else doc.metadata.get("end_line", -1),
                doc.page_content,
                doc.metadata.get("span_coord", "char"),
            )
            for doc in docs
        ]
        for docs in all_context
    ]

    # Build serialized_context using merged interval text.
    # _normalise_retrieved_spans merges nearby spans from the same file (gap=80
    # lines) and returns the actual file text covering all merged spans — so the
    # LLM receives a contiguous, complete context instead of fragmented children.
    serialized_context = []
    for spans_raw in retrieved_spans:
        _, merged_text = _normalise_retrieved_spans(spans_raw)
        serialized_context.append(merged_text)

    batch["query_id"] = [str(query_id) for query_id in batch["query_id"]]

    enc = tiktoken.get_encoding("gpt2")

    prompts = []
    for question, context in zip(batch["query"], serialized_context):
        # Reserve tokens for fixed prompt parts; truncate only the context
        PROMPT_OVERHEAD = len(enc.encode(
            SYSTEM_INSTRUCTIONS + "\n\n" +
            f"Question: {question}\n\n Context:\n\n"
            # f"Now answer the question correctly, completely and faithfully :"
        ))
        context_budget    = max(MAX_TOKENS - PROMPT_OVERHEAD, 100)
        context_tokens    = enc.encode(context)
        truncated_context = enc.decode(context_tokens[:context_budget])

        user_content = (
            f"Question: {question}\n\n"
            f"Context: {truncated_context}\n\n"
            f"Answer:"
        )
        prompts.append([
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user",   "content": user_content},
        ])

    return {
        "prompts":            prompts,
        "retrieved_spans":    retrieved_spans,
        "serialized_context": serialized_context,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground-truth fields after generation.

    Adds to batch
    -------------
    ground_truth_spans  list[list[tuple[str, int, int]]]
        Raw (file, start_line, end_line) tuples from qrels.
        to_spans validation happens in compute_metrics_fn.
    """
    ground_truth_spans = []
    for query_id in batch["query_id"]:
        rows = qrels[qrels["query_id"] == str(query_id)]
        # qrels always stores real line numbers (built in cell 6)
        raw = [
            (row["file"], int(row["start_line"]), int(row["end_line"]))
            for _, row in rows.iterrows()
        ]
        ground_truth_spans.append(raw)  # to_spans called in compute_metrics_fn

    batch["ground_truth_spans"] = ground_truth_spans

    assert "reference_answer" in batch, (
        "reference_answer missing from batch — make sure it is included when "
        "building the queries Dataset in cell 6."
    )
    return batch


### Define Custom Eval Metrics Functions

In [58]:
# All retrieval metrics delegate to project1_eval so scoring is identical to
# the TA's evaluate_retrieval.py and run_judge.py scripts.

JUDGE_MODEL    = "api-gpt-oss-120b"    #"claude-sonnet-4-6"   # "api-gpt-oss-120b"   
JUDGE_BASE_URL = "https://tritonai-api.ucsd.edu"
K = 5

# Module-level cache: source filename → full file text.
# Populated lazily on first access; persists across batches within a session.
_file_line_cache: dict[str, str] = {}


SOURCEDOCS_PATH_EVAL = Path("/home/yup031/Project1/sourcedocs/")


def _get_file_lines(fname: str) -> list[str]:
    """Return all lines of a source file, cached."""
    if fname not in _file_line_cache:
        candidates = list(SOURCEDOCS_PATH_EVAL.rglob(fname))
        if not candidates:
            return []
        _file_line_cache[fname] = candidates[0].read_text(encoding="utf-8")
    return _file_line_cache[fname].splitlines()


def _content_to_lines(fname: str, content: str) -> tuple[int, int]:
    """Find chunk content in source file, return 1-based (start_line, end_line)."""
    file_lines = _get_file_lines(fname)
    if not file_lines:
        return 1, 1
    text   = "\n".join(file_lines)
    needle = content.strip()
    pos    = text.find(needle)
    if pos == -1:
        first_line = next((l.strip() for l in needle.splitlines() if l.strip()), "")
        pos = text.find(first_line) if first_line else -1
    if pos == -1:
        return 1, 1
    start_line = text[:pos].count("\n") + 1
    end_line   = text[:pos + len(needle)].count("\n") + 1
    return start_line, end_line


def _merge_spans_by_proximity(
    spans: list[tuple[str, int, int]],
    gap: int = 80,
) -> list[tuple[str, int, int, str]]:
    """Merge spans from the same file that are within `gap` lines of each other.

    Returns list of (file, start_line, end_line, merged_text) tuples where
    merged_text is the actual file content covering the merged span.
    """
    if not spans:
        return []

    # Group by file
    by_file: dict[str, list[tuple[int, int]]] = {}
    for fname, start, end in spans:
        by_file.setdefault(fname, []).append((start, end))

    result = []
    for fname, intervals in by_file.items():
        intervals.sort(key=lambda x: x[0])

        # Merge overlapping or nearby intervals
        merged: list[tuple[int, int]] = []
        cur_start, cur_end = intervals[0]
        for start, end in intervals[1:]:
            if start - cur_end <= gap:
                cur_end = max(cur_end, end)
            else:
                merged.append((cur_start, cur_end))
                cur_start, cur_end = start, end
        merged.append((cur_start, cur_end))

        # Extract actual file text for each merged span.
        # No clamping — the span and the text fed to the LLM must match exactly.
        file_lines = _get_file_lines(fname)
        for start_line, end_line in merged:
            s    = max(0, start_line - 1)        # 0-based inclusive
            e    = min(len(file_lines), end_line) # 0-based exclusive
            text = "\n".join(file_lines[s:e])
            result.append((fname, start_line, end_line, text))

    return result


def _normalise_retrieved_spans(
    raw_spans: list[tuple[str, int, int, str, str]],
) -> tuple[list[tuple[str, int, int]], str]:
    """Convert raw spans to line numbers, merge nearby spans within same file.

    raw_spans elements: (fname, start, end, content, span_coord)

    Returns:
        spans_for_scoring  list[(file, start_line, end_line)] — validated by to_spans
        merged_text        str — concatenated text of all merged intervals for LLM
    """
    # Step 1: locate each chunk in its source file → exact line numbers
    converted = []
    for fname, start, end, content, coord in raw_spans:
        start_line, end_line = _content_to_lines(fname, content)
        converted.append((fname, start_line, end_line))

    # Step 2: merge nearby spans within same file (gap ≤ 80 lines)
    merged = _merge_spans_by_proximity(converted, gap=80)

    # Step 3: spans for scoring (strip text, validate via to_spans)
    spans_for_scoring = to_spans([(f, s, e) for f, s, e, _ in merged])

    # Step 4: full merged text for LLM context
    merged_text = "\n\n".join(text for _, _, _, text in merged)

    return spans_for_scoring, merged_text


def _normalise_gt_spans(
    raw_spans: list[tuple[str, int, int]],
) -> list[tuple[str, int, int]]:
    """Validate and deduplicate ground-truth spans through project1_eval.to_spans.

    qrels always stores real line numbers so no coordinate conversion needed.
    """
    return to_spans(raw_spans)


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval and generation metrics for one batch.

    Reads from batch fields:
        retrieved_spans    list[list[tuple[str, int, int, str]]]  (with span_coord)
        ground_truth_spans list[list[tuple[str, int, int]]]       (always line numbers)
        serialized_context list[str]
        generated_text     list[str]
        reference_answer   list[str]
        query              list[str]
    """
    total_queries = len(batch["query"])

    precisions, recalls, f1s = [], [], []
    correctness_scores, faithfulness_scores, completeness_scores = [], [], []
    judge_failures = 0

    for pred_spans_raw, gt_spans_raw, context, gen_text, ref_ans, query in zip(
        batch["retrieved_spans"],
        batch["ground_truth_spans"],
        batch["serialized_context"],
        batch["generated_text"],
        batch["reference_answer"],
        batch["query"],
    ):
        # Convert + validate here, just before scoring
        pred_spans, _ = _normalise_retrieved_spans(pred_spans_raw)
        gt_spans   = _normalise_gt_spans(gt_spans_raw)

        precisions.append(precision_at_k(pred_spans, gt_spans, k=K))
        recalls.append(recall_at_k(pred_spans,       gt_spans, k=K))
        f1s.append(f1_at_k(pred_spans,               gt_spans, k=K))

        judge = call_judge(
            query=query,
            reference_answer=ref_ans,
            retrieved_context=context,
            system_answer=gen_text,
            model=JUDGE_MODEL,
            base_url=JUDGE_BASE_URL,
            api_key=TRITON_API_KEY,
        )
        if judge.get("failed"):
            judge_failures += 1
        correctness_scores.append(judge["correctness"])
        faithfulness_scores.append(judge["faithfulness"])
        completeness_scores.append(judge["completeness"])

    def mean(xs): return sum(xs) / len(xs) if xs else 0.0

    avg_precision    = mean(precisions)
    avg_recall       = mean(recalls)
    avg_f1           = mean(f1s)
    avg_correctness  = mean(correctness_scores)
    avg_faithfulness = mean(faithfulness_scores)
    avg_completeness = mean(completeness_scores)

    retrieval_score   = (avg_f1 + avg_precision + avg_recall) / 3
    generation_score  = (avg_correctness + avg_faithfulness + avg_completeness / 5) / 3
    leaderboard_score = 0.5 * retrieval_score + 0.5 * generation_score

    return {
        "Total":          {"value": total_queries},
        f"Precision_at{K}": {"value": mean(precisions)},
        f"Recall_at{K}":    {"value": mean(recalls)},
        f"F1_at{K}":        {"value": mean(f1s)},
        "Correctness":    {"value": mean(correctness_scores)},
        "Faithfulness":   {"value": mean(faithfulness_scores)},
        "Completeness":   {"value": mean(completeness_scores)},
        "JudgeFailures":  {"value": judge_failures},
        "LeaderboardScore": {"value": leaderboard_score},
        "RetrievalScore": {"value": retrieval_score},
        "GenerationScore": {"value": generation_score},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Weighted average of per-batch metrics across all shards."""
    num_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total = sum(num_per_batch)

    weighted_keys = [
        f"Precision_at{K}", f"Recall_at{K}", f"F1_at{K}",
        "Correctness", "Faithfulness", "LeaderboardScore",
        "RetrievalScore", "GenerationScore"
    ]
    result = {"Total": {"value": total}}

    for key in weighted_keys:
        result[key] = {
            "value": sum(
                m["value"] * q
                for m, q in zip(aggregated_metrics[key], num_per_batch)
            ) / total,
            "is_algebraic": True,
            "value_range": (0, 1),
        }

    result["Completeness"] = {
        "value": sum(
            m["value"] * q
            for m, q in zip(aggregated_metrics["Completeness"], num_per_batch)
        ) / total,
        "is_algebraic": True,
        "value_range": (0, 5),
    }

    result["JudgeFailures"] = {
        "value": sum(m["value"] for m in aggregated_metrics["JudgeFailures"]),
        "is_algebraic": False,
    }

    return result


### Define Partial Multi-Config Knobs for OpenAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [59]:
# Generator model config (shared across all RAG variants)
_GENERATOR_CLIENT_CFG = {
    "api_key":     TRITON_API_KEY,
    "base_url":    "https://tritonai-api.ucsd.edu",
    "max_retries": 2,
}
_GENERATOR_MODEL_CFG = {
    "model": "claude-sonnet-4-6",   
    "max_completion_tokens": 3000,
}

def _make_openai_config(rag_variant: RFLangChainRagSpec) -> RFOpenAIAPIModelConfig:
    return RFOpenAIAPIModelConfig(
        client_config=_GENERATOR_CLIENT_CFG,
        model_config=_GENERATOR_MODEL_CFG,
        rpm_limit=120,
        tpm_limit=1_000_000,
        rag=rag_variant,
        prompt_manager=None,
    )

# One entry per RAG strategy; the grid search will run them in parallel.
# Each entry is labelled so results_df rows are easy to identify.
config_set = List([
    {
        "openai_config":          _make_openai_config(rag_variant),
        "batch_size":             batch_size,
        "preprocess_fn":          openai_sample_preprocess_fn,
        "postprocess_fn":         sample_postprocess_fn,
        "compute_metrics_fn":     sample_compute_metrics_fn,
        "accumulate_metrics_fn":  sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name":    "normal",
            "confidence_level": 0.95,
            "use_fpc":          True,
        },
        # Label surfaces in results_df so you can tell the strategies apart
        "config_label": label,
    }
    for rag_variant, label in all_splitters
])


### Create Config Group

In [60]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

In [61]:
# from torch.utils.tensorboard import SummaryWriter

# tensorboard_log_dir = f"/home/yup031/rapidfireai/rapidfire_experiments/{my_experiment}/tensorboard_logs"
# writer = SummaryWriter(log_dir=tensorboard_log_dir)
# writer.add_scalar("test", 0.0, 0)
# writer.close()

# %tensorboard --logdir {tensorboard_log_dir}

### Run Multi-Config Evals

In [ ]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks
results = experiment.run_evals(
    config_group=config_group,
    dataset=queries,
    num_shards=4,
    seed=42,
)


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,40.2s,CPU,FAISS
2,Complete,40.3s,CPU,FAISS
3,Complete,40.3s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.check_embedding_ctx_length,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.k,search_cfg.type,model_config
1,claude-sonnet-4-6,STOPPED,0/4,-,LineNumberSplitter,128,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,max_completion_tokens=3000
2,claude-sonnet-4-6,ONGOING,0/4,-,LineNumberSplitter,256,128,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,max_completion_tokens=3000
3,claude-sonnet-4-6,ONGOING,0/4,-,StructuralRSTSplitter,256,0,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,max_completion_tokens=3000


### View Results

In [ ]:
# Convert results dict to DataFrame – one row per config, one column per metric
results_df = pd.DataFrame([
    {
        "run_id": run_id,
        **{
            k: (v["value"] if isinstance(v, dict) and "value" in v else v)
            for k, v in metrics_dict.items()
        },
    }
    for run_id, (_, metrics_dict) in results.items()
])

# Reorder columns: retrieval metrics first, then judge metrics
retrieval_cols = [c for c in results_df.columns if any(
    c.startswith(p) for p in ["Precision", "Recall", "F1"])]
judge_cols     = ["Correctness", "Faithfulness", "Completeness", "JudgeFailures","LeaderboardScore"]
other_cols     = [c for c in results_df.columns
                  if c not in retrieval_cols + judge_cols + ["run_id", "Total"]]

results_df = results_df[
    ["run_id", "Total"] + retrieval_cols + judge_cols + other_cols
]
results_df


KeyError: "None of [Index(['run_id', 'Total', 'Correctness', 'Faithfulness', 'Completeness',\n       'JudgeFailures', 'LeaderboardScore'],\n      dtype='object')] are in the [columns]"

In [ ]:
import os

log_dir = f"/home/yup031/Project1/logs"
os.makedirs(log_dir, exist_ok=True)

output_path = f"{log_dir}/results.csv"

file_exists = os.path.exists(output_path)
results_df.to_csv(output_path, mode="a", header=not file_exists, index=False)
print(f"{'Appended to' if file_exists else 'Created'} {output_path}")

Appended to /home/yup031/Project1/logs/results.csv


### End Experiment

In [ ]:
experiment.end()

Experiment exp_0508_1 ended


### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/yup031/rapidfireai/logs/exp_0508_1/rapidfire.log

Last 30 lines of rapidfire.log:
2026-05-08 14:14:01 | DocProcessingActor | INFO | doc_actor.py:59 | [exp_0508_1:DocProcessingActor] DocProcessingActor initialized
2026-05-08 14:14:03 | DocProcessingActor | INFO | doc_actor.py:96 | [exp_0508_1:DocProcessingActor] DocProcessingActor: Starting context initialization...
2026-05-08 14:14:03 | DocProcessingActor | INFO | doc_actor.py:107 | [exp_0508_1:DocProcessingActor] Building document index...
2026-05-08 14:14:08 | DocProcessingActor | INFO | doc_actor.py:109 | [exp_0508_1:DocProcessingActor] Document index built successfully
2026-05-08 14:14:08 | DocProcessingActor | INFO | doc_actor.py:134 | [exp_0508_1:DocProcessingActor] Serializing FAISS index for cross-actor sharing...
2026-05-08 14:14:08 | DocProcessingActor | INFO | doc_actor.py:144 | [exp_0508_1:DocProcessingActor] FAISS index serialized: 2730492 bytes
2026-05-08 14:14:08 | DocProcessingActor | INFO | doc_actor.